# Data Validation & Data Quality Checks

## Definition
**Data validation** verifies that your data meets expected constraints before it enters an ML pipeline.
Poor data quality = poor model performance. Garbage in, garbage out.

### Types of Checks
| Check Type | Examples |
|------------|---------|
| **Type checks** | Age must be int, email must be str |
| **Range checks** | Age 0–120, probability 0–1 |
| **Null checks** | Required fields must not be null |
| **Uniqueness** | IDs must be unique |
| **Consistency** | End date > Start date |
| **Cardinality** | Categorical column has expected values |
| **Distribution** | Target column not 100% one class |
| **Referential integrity** | FK values must exist in related table |

In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    'id':       [1, 2, 3, 2, 5],         # duplicate!
    'age':      [25, -5, 30, 200, 28],    # -5 and 200 invalid!
    'email':    ['a@b.com', 'invalid', 'c@d.com', 'e@f.com', None],
    'income':   [50000, 60000, np.nan, 80000, 70000],
    'category': ['A', 'B', 'Z', 'A', 'C'], # 'Z' not in expected ['A','B','C']
    'score':    [0.85, 0.72, 0.91, 1.5, 0.68]  # 1.5 out of range [0,1]!
})
print(df)

In [ ]:
# 1. NULL checks
def check_nulls(df, required_cols):
    print('=== NULL CHECK ===')
    for col in required_cols:
        null_count = df[col].isna().sum()
        status = 'FAIL' if null_count > 0 else 'PASS'
        print(f'  [{status}] {col}: {null_count} nulls')

check_nulls(df, ['id', 'email', 'income'])

In [ ]:
# 2. Uniqueness checks
def check_unique(df, unique_cols):
    print('=== UNIQUENESS CHECK ===')
    for col in unique_cols:
        dups = df[col].duplicated().sum()
        status = 'FAIL' if dups > 0 else 'PASS'
        print(f'  [{status}] {col}: {dups} duplicates')

check_unique(df, ['id'])

In [ ]:
# 3. Range checks
def check_range(df, col, lo, hi):
    violations = ((df[col] < lo) | (df[col] > hi)).sum()
    status = 'FAIL' if violations > 0 else 'PASS'
    print(f'  [{status}] {col} in [{lo}, {hi}]: {violations} violations')
    return df[(df[col] < lo) | (df[col] > hi)]

print('=== RANGE CHECK ===')
bad_age   = check_range(df, 'age', 0, 120)
bad_score = check_range(df, 'score', 0.0, 1.0)
print('Bad ages:', bad_age['age'].values)
print('Bad scores:', bad_score['score'].values)

In [ ]:
# 4. Cardinality / allowed values check
def check_cardinality(df, col, allowed_values):
    unexpected = ~df[col].isin(allowed_values)
    count = unexpected.sum()
    status = 'FAIL' if count > 0 else 'PASS'
    print(f'  [{status}] {col} allowed values: {count} unexpected')
    if count:
        print(f'    Unexpected: {df.loc[unexpected, col].unique().tolist()}')

print('=== CARDINALITY CHECK ===')
check_cardinality(df, 'category', ['A', 'B', 'C'])

In [ ]:
# 5. Email format validation
import re
def is_valid_email(email):
    if pd.isna(email): return False
    pattern = r'^[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}$'
    return bool(re.match(pattern, email))

df['email_valid'] = df['email'].apply(is_valid_email)
print('=== EMAIL VALIDATION ===')
print(df[['email', 'email_valid']])

In [ ]:
# 6. Comprehensive validation report
def validate_dataframe(df):
    report = []
    report.append(('shape', df.shape, 'INFO'))
    for col in df.columns:
        null_pct = df[col].isna().mean() * 100
        if null_pct > 0:
            report.append((col, f'{null_pct:.1f}% null', 'WARN' if null_pct < 10 else 'FAIL'))
    if df.duplicated().sum() > 0:
        report.append(('duplicates', df.duplicated().sum(), 'FAIL'))
    print(f'{"Field":20s} {"Issue":30s} {"Status":6s}')
    print('-' * 60)
    for field, issue, status in report:
        print(f'{str(field):20s} {str(issue):30s} {status:6s}')

validate_dataframe(df)